# Developer 1 — ML baseline walkthrough

This notebook records the current grouped-v1 baseline setup, fixed preprocessing, environment, and saved validation result. It does **not** train, download weights, open dataset images, run inference, or calculate test metrics. Its code cells are safe to use with Run All.

The ALL-class output is a model score, not a calibrated leukemia probability. This is a research prototype, not a diagnostic system.


## 1. Locate the ML project

In [ ]:
from pathlib import Path
import json
import platform
import sys

start = Path.cwd().resolve()
candidates = [start, start / "ml", *start.parents, *(parent / "ml" for parent in start.parents)]
ML_ROOT = next(
    (path for path in candidates
     if (path / "configs/baseline-grouped-v1.yaml").is_file()
     and (path / "src").is_dir()),
    None,
)
if ML_ROOT is None:
    raise FileNotFoundError("Cannot locate ml/configs/baseline-grouped-v1.yaml.")
ML_ROOT = ML_ROOT.resolve()
REPO_ROOT = ML_ROOT.parent
sys.path.insert(0, str(ML_ROOT))
print(f"Repository: {REPO_ROOT}")
print(f"ML folder: {ML_ROOT}")
print(f"Python: {platform.python_version()}")


## 2. Check the environment

This reports installed versions and MPS availability. It does not create a model or start training.

In [ ]:
import importlib.metadata as package_metadata
import torch
import torchvision
import yaml

for package in ("torch", "torchvision", "numpy", "Pillow", "PyYAML", "scikit-learn"):
    try:
        print(f"{package}: {package_metadata.version(package)}")
    except package_metadata.PackageNotFoundError:
        print(f"{package}: not installed")
print(f"MPS built: {torch.backends.mps.is_built()}")
print(f"MPS available: {torch.backends.mps.is_available()}")


## 3. Inspect the fixed configuration and split

This notebook follows the newer frozen grouped-v1 development split. It preserves source subject IDs and assigns reviewed dependency groups wholly to one partition. Those groups are conservative image-relationship groupings, not verified patient identities.

Raw data, the split verification file, and trained artifacts are local and Git-ignored. If a teammate does not have these local files, this notebook reports that and does not download them.

In [ ]:
import yaml

with (ML_ROOT / "configs/baseline-grouped-v1.yaml").open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
dataset_config = config["dataset"]
model_config = config["model"]
training_config = config["training"]
frozen = dataset_config["frozen_split"]

print("Model:", model_config["architecture"], "|", model_config["version"])
print("Image size:", model_config["image_size"], "square pixels")
print("Pretrained ImageNet weights:", model_config["pretrained"])
print("Backbone frozen for:", model_config["freeze_backbone_epochs"], "epochs")
print("Training settings:", {
    key: training_config[key]
    for key in ("seed", "epochs", "batch_size", "learning_rate", "weight_decay", "device", "num_workers")
})
print("Frozen split SHA-256:", frozen["sha256"])
print("Grouping policy:", frozen["policy_version"])
print("Training manifest present:", (ML_ROOT / dataset_config["manifest"]).is_file())
print("Image folder present:", (ML_ROOT / dataset_config["image_root"]).is_dir())

verification_path = ML_ROOT / frozen["verification_path"]
if verification_path.is_file():
    verification = json.loads(verification_path.read_text(encoding="utf-8"))
    assert verification["split_manifest_sha256"] == frozen["sha256"]
    print("\nFrozen split verification matches the config.")
    for partition in ("train", "validation"):
        counts = verification["counts"][partition]
        class_counts = counts["class_images"]
        print(
            f"{partition}: {counts['images']:,} images; "
            f"ALL={class_counts['all']:,}, HEM={class_counts['hem']:,}; "
            f"{counts['subjects']} subject IDs; {counts['dependency_groups']} dependency groups"
        )
    print("Test images are intentionally not summarized or scored here.")
else:
    print("Local split verification not found. Supply the dataset workspace before training.")


## 4. Confirm preprocessing

Evaluation and inference share the same resize and normalization. Training adds the two listed augmentations. This cell only prints metadata; it does not read an image.

In [ ]:
from src.preprocessing import PreprocessingMetadata

preprocessing = PreprocessingMetadata(image_size=model_config["image_size"]).to_dict()
print(json.dumps(preprocessing, indent=2))
print("Training-only augmentations: horizontal flip (p=0.5), rotation up to 5 degrees.")
print("Evaluation/inference augmentations: none.")


## 5. Review the saved validation result

This reads existing JSON summaries only. It does not load weights, access images, run inference, or inspect test predictions.

In [ ]:
metadata_path = ML_ROOT / "artifacts/baseline-grouped-v1/model-metadata.json"
validation_path = ML_ROOT / "artifacts/diagnostics/grouped-v1-mps/summary.json"

if metadata_path.is_file() and validation_path.is_file():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    result = json.loads(validation_path.read_text(encoding="utf-8"))
    assert result["partition"] == "validation"
    assert result["test_images_evaluated"] == 0
    assert metadata["test_evaluated"] is False and metadata["test_metrics"] is None
    assert metadata["checkpoint_sha256"] == result["checkpoint_sha256"]
    assert metadata["split_manifest_sha256"] == frozen["sha256"]
    assert metadata["preprocessing"] == preprocessing
    metrics = result["metrics"]
    print("Model:", metadata["model_version"])
    print("Checkpoint SHA-256:", metadata["checkpoint_sha256"])
    print("Validation images:", metrics["sample_count"])
    print(f"Weighted F1: {metrics['weighted_f1']:.2%}")
    print(f"Sensitivity: {metrics['sensitivity']:.2%}")
    print(f"Specificity: {metrics['specificity']:.2%}")
    print(f"Validation loss: {metrics['mean_loss']:.4f}")
    print("Confusion matrix (actual rows [ALL, HEM], predicted columns [ALL, HEM]):")
    for row in metrics["confusion_matrix"]:
        print(row)
    print("Test evaluation: not performed.")
else:
    print("Saved baseline artifacts are local and not present in this checkout.")


## 6. Review validation progress by epoch

The checkpoint is selected by lowest validation loss. The following cell reads saved history; it does not retrain.

In [ ]:
history_path = ML_ROOT / "artifacts/baseline-grouped-v1/history.jsonl"
if history_path.is_file():
    history = [
        json.loads(line)
        for line in history_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("epoch | val loss | weighted F1 | sensitivity | specificity")
    for entry in history:
        metrics = entry["validation_metrics"]
        print(
            f"{entry['epoch']:>5} | {metrics['mean_loss']:>8.4f} | "
            f"{metrics['weighted_f1']:>11.2%} | {metrics['sensitivity']:>11.2%} | "
            f"{metrics['specificity']:>11.2%}"
        )
    best = min(history, key=lambda entry: entry["validation_metrics"]["mean_loss"])
    print("Saved checkpoint epoch:", best["epoch"])
else:
    print("Saved history is not present in this checkout.")


## 7. Future training is manual

Do not run the original grouped-v1 config again: its output directory already contains a checkpoint, and the training script protects it from overwriting. For a new experiment, copy the config, set a new model version and a fresh output directory, then run the data-only preflight.

Commands below are examples only. This notebook does not execute them; you decide when to run training.

~~~sh
cd /Users/johndavidvelos/Documents/HemoScopeAI
source .venv/bin/activate
cd ml
# Create configs/YOUR_NEW_CONFIG.yaml with a unique artifacts/YOUR_RUN output_dir first.
python -u -m src.train --config configs/YOUR_NEW_CONFIG.yaml --check-data-only
# Run this yourself only when ready to train:
caffeinate -i python -u -m src.train --config configs/YOUR_NEW_CONFIG.yaml
~~~

To reproduce the figures later, install the optional analysis extra from the ml/ folder.

## Developer 1 checklist

This notebook fills the missing Day 1 environment/baseline-notebook artifact and documents fixed preprocessing and saved validation results. The local notebook is ready. Open it and run the safe inspection cells once in Jupyter to confirm they work in your setup. Training and test evaluation remain separate.